# ‘Pent-Up’ Frustration 3 / Knight Moves 7

**Jane Street puzzle, July 2026** — [puzzle page](https://www.janestreet.com/puzzles/pent-up-frustration-3-knight-moves-7-index/)

> **AI use:** I solved this puzzle with AI assistance, specifically I had AI build a sandbox tool to use similar to sudokupad.  I then made logical deductions and tried to solve the entire puzzle by hand.  This proved quite doable (surprising), but I also had AI make me a tooling agent to help me enumerate the math pathways which made figuring out the series of jump ups, jump downs, and jump flats a lot easier once the 3 move rule was dispelled.  I eventually got stuck, but by then had reduced that search space so much the final result took 20 milliseconds to query.  I'll try to walk through my logic starting from early assumptions.  I had AI help me with this writeup too — mostly so I wouldn't have to format all the notation in markdown which is tedious.  I think AI could have easily one-shotted this puzzle to be honest, but it was very fun to solve logically so I'm glad I put up guardrails and gave it a shot myself.
>
> This notebook was assembled from this folder's original markdown files (`PUZZLE.md`, `README.md`, `SOLUTION.md`) by an AI assistant; the write-up is my own.

## Puzzle

Source: https://www.janestreet.com/puzzles/pent-up-frustration-3-knight-moves-7-index/
Image:  https://www.janestreet.com/puzzles/pent-3-knight-7.png  (855x855)

### Rules (verbatim)

The board above has been tiled with the 12 pentominoes (plus a 2-by-2 tetromino) into 13
regions. Think of each of these 13 regions as constructed out of 1-by-1-by-1 cubes. We need
to add a tower to each region. A tower is an additional size-1 cube placed on one of a
region's squares.

After adding these towers, place a knight at the bottom-left square. It then proceeds to
make knight's moves until it has visited all the towers. It never visits the same space
twice. (A move on this board involves travelling 0 units in one dimension, 1 in another,
and 2 in the third. The knight is allowed to "pass through" towers as it moves.)

But there's a catch: As you can see, the knight starts with a score of 0. On its Nth move,
its score increases by N if the move is to a location at the same altitude as the square it
moved from. If, instead, it moves up, the score is multiplied by N. And finally, if it moves
down, the score is divided by N. This last type of move is only allowed if the score is
evenly divisible by N.

Every three moves, up until move #18, the knight wrote down its score upon arriving at a
given square. From then on it only wrote down its score every K moves, for some larger value
K. Using this information, can you reconstruct the knight's path?

After filling all the remaining visited squares with the missing score values, find the
unvisited squares. For each of these squares, compute the sum of the scores in any
orthogonally adjacent squares that were part of the knight's path. The answer to this puzzle
is the sum of these "neighbor sums" from the unvisited squares.

### Board (a1 = bottom-left = the "0" start square)

Letters are the pentomino name of each region; O is the 2x2 tetromino.

```
      a  b  c  d  e  f  g  h
  8   I  I  I  I  I  V  V  V     8
  7   U  U  U  P  P  Z  Z  V     7
  6   U  N  U  P  P  P  Z  V     6
  5   Y  N  N  O  O  F  Z  Z     5
  4   Y  Y  N  O  O  F  F  L     4
  3   Y  T  N  X  F  F  W  L     3
  2   Y  T  X  X  X  W  W  L     2
  1   T  T  T  X  W  W  L  L     1
      a  b  c  d  e  f  g  h
```

Given scores (12 cells):

    a1 = 0 (start)   a5 = 528   b4 = 449   b3 = 750
    d6 = 23          f6 = 138   d3 = 88    e4 = 16
    f8 = 37          h8 = 1100  g3 = 1     f3 = 272

#### How this was extracted

Rendered the PNG to a canvas and measured pixels rather than eyeballing:

- Grid lines at 48.5, 143.5, ..., 808.5 -> 8x8, cell size exactly 95px.
- Every interior edge measured: thin = 2px, thick = 7px, no ambiguous values.
- Flood fill with those walls -> 13 regions, sizes 5x12 + 4, total 64.
- Shape check: the 13 regions are exactly the 12 distinct free pentominoes
  (I V U P Z N Y F L T X W) plus one 2x2 tetromino. A single mis-read wall would
  almost certainly have produced a duplicate shape or a non-pentomino, so this is
  strong independent confirmation.
- Scanned each cell interior for ink -> exactly 12 cells contain numbers.

### Derived move set

The board is one cube thick, so a square's standable altitude is 0 (plain) or 1 (tower).
A move's displacement (dx, dy, dz) is a signed permutation of (0, 1, 2). Since |dz| <= 1,
the dz = +/-2 case is impossible. That leaves exactly two families:

| Family | Displacement                    | Altitude        | Score on move N         |
|--------|---------------------------------|-----------------|-------------------------|
| Level  | ordinary knight move, dz = 0    | both ends equal | score += N              |
| Up     | straight 2 squares orthogonally | plain -> tower  | score *= N              |
| Down   | straight 2 squares orthogonally | tower -> plain  | score /= N if divisible |

Consequences:

- A knight on a tower can make an ordinary knight move only to another tower.
- Up/down moves are rook-like two-steppers, not knight moves.
- Score starts at 0, and 0 is divisible by everything, so early down-moves are legal
  and leave the score at 0.
- Each square has exactly one standable altitude, so "never visits the same space twice"
  reduces to never revisiting a square.

### Derived checkpoint structure (tool has this OFF by default)

A square shows a number iff the knight was on it at a recording moment. Recording moments
are moves 0, 3, 6, 9, 12, 15, 18, then 18+K, 18+2K, ... for some K > 3.

12 clue cells = 7 early records (including move 0) + 5 late records, so the path reaches at
least move 18+5K. The path visits M+1 <= 64 distinct squares, so M <= 63, giving K <= 9.

| K | total moves M must be in |
|---|--------------------------|
| 4 | 38-41 |
| 5 | 43-47 |
| 6 | 48-53 |
| 7 | 53-59 |
| 8 | 58-63 |
| 9 | 63 exactly |

The first clue cell hit after move 18 pins K, after which every remaining checkpoint move
number is fixed.

## Solution

**Answer: 33609**

### Notation

Every square on the board carries **three** attributes, and the walkthrough below refers to
each of them constantly, so it's worth fixing the shorthand up front.

| attribute | means | written as |
|---|---|---|
| **location** | where it is — file `a`–`h`, rank `1`–`8` | `f6` |
| **move number** | *when* the knight stands there | `#25` |
| **value** | the running score on arrival | `138` |

Written together, in that order:

> **`f6 #25 = 138`** — “the knight is on `f6` on move 25, holding a score of 138.”

Any attribute can be dropped when it isn't known yet, which happens constantly while solving:

- **`f6 = 138`** — a printed clue. The board tells us the value; the move number is still open.
- **`#25 = 138`** — the score chain has pinned the value of a move, but not its square.
- **`f6 #25`** — the square and its move are pinned, value implied.

Two more marks:

- **`▲`** after a location means that square is a **tower** (altitude 1). `c4▲ #21 = 2667`.
  No mark means ground (altitude 0).
- Moves are written by their effect: **`+21`**, **`×21`**, **`÷21`** for a level, up or down
  move made *on move 21*. The number in the operator is always the move number, which is what
  makes the arithmetic tick along.

Regions are named by their pentomino letter — **I V U P Z N Y F L T X W** — plus **O** for the
2×2 tetromino. `f7[Z]` means “`f7`, which lives in the Z pentomino.”

So a full leg of the path reads:

> `d3 #18 = 88` → `+19` → `b2 #19 = 107` → `+20` → `a4 #20 = 127` → `×21` → `c4▲ #21 = 2667`

### Key early deductions

**1. There are only three kinds of move**

A move's displacement `(dx, dy, dz)` is a signed permutation of `(0, 1, 2)`. The board is one
cube thick, so a square's standable altitude is 0 (ground) or 1 (tower) — which makes
`dz = ±2` impossible. Only two families of permutation therefore survive.

| move | shape | altitude | score on move N |
|---|---|---|---|
| **level** | ordinary knight move | both ends equal | `+N` → `score + N` |
| **up** | straight 2 orthogonally | ground → tower | `×N` → `score × N` |
| **down** | straight 2 orthogonally | tower → ground | `÷N` → `score ÷ N`, only if it divides |

Two consequences do most of the work later in determining which move is made when. A knight standing on a tower can only make an
ordinary knight move **to another tower**. And `×` and `÷` must alternate, with any run of `+`
between them — never two multiplies or two divides in a row.  This was later used to figure out the value of `K` as well as how to jump between marked candidates.

**2. A square shows a number *if and only if* the knight was on it at a recording moment.**

Recording moments are moves 0, 3, 6, …, 18, then every `K` moves. Twelve clue cells means
7 early records plus **5** late ones, so the path reaches move `18 + 5K`. It visits `M+1 ≤ 64`
distinct squares, so `M ≤ 63`, giving **`4 ≤ K ≤ 9`**. Equally useful in the other direction:
landing on a clue cell at a *non*-recording move is illegal, which kills candidate routes fast.

### Checkpoint 1: moves 1–3 are forced

I was quickly able to key in on the `g3 = 1` clue as impossible other than as a first move.  The start square and the next two are all towers since we need to jump down after adding 1 and 2, and since each region holds exactly one tower, there is only one way to place them:

> `a1▲ #0 = 0` → `+1` → `c2▲ #1 = 1` → `+2` → `e3▲ #2 = 3` → `÷3` → `g3 #3 = 1`

![First Checkpoint](SS1.png)

This is a very useful and restrictive start! Three towers placed — `a1[T]`, `c2[X]`, `e3[F]`.

### Checkpoint 2

Checkpoint 2 requires figuring out what we can do with 4, 5, and 6 to get to a candidate square.  Pretty quickly you can enumerate your options and realize that `16` is the only arrivable square, which requires 3 regular knight jumps.  We don't yet know how we get there, but move 6 lands us on `e4 #6 = 16`.

> `g3 #3 = 1` → `+4` → `+5` → `+6` → `e4 #6 = 16`

![Second Checkpoint](SS2.png)

### Checkpoint 3

A similar process lets us realize checkpoint 3 must be `23`, by doing a `×7`, `÷8`, `+9` route with a tower being hit on move 7. The only legal routing for these steps is shown below.

> `e4 #6 = 16` → `×7` → `e6▲ #7 = 112` → `÷8` → `e8 #8 = 14` → `+9` → `d6 #9 = 23`

![Third Checkpoint](SS3.png)

### Checkpoint 4

Again, enumeration is fairly easy with this restrictive 3 move rule, so we arrive at `528` as the only candidate landing spot for move 12, achievable by `+10`, `+11`, `×12`.  This places a tower at `a5`.  The pathing to `a5` is unknown.

> `d6 #9 = 23` → `+10` → `+11` → `×12` → `a5▲ #12 = 528`

![Fourth Checkpoint](SS4.png)

### Checkpoint 5

A very useful and restrictive checkpoint.  We need to do `+13`, `+14`, `÷15` to reach `f8 #15 = 37`.  These 2 adds have to be done at level 1, so we get to place 2 more towers in the grid.  Additionally, the only way to reach `f8` is from `d8`, `f6`, and `h8` (a `÷` move travels 2 in a straight line, so those are its three straight-2 neighbours), and we can't reach `f6` or `h8` from `a5`.  We reach `d8` via `b7` or `c6`, so we know a tower exists at one of those two locations, which I've marked with a red `?`.

> `a5▲ #12 = 528` → `+13` → `(b7▲ or c6▲) #13 = 541` → `+14` → `d8▲ #14 = 555` → `÷15` → `f8 #15 = 37`

Both `b7` and `c6` are in region **U**, so whichever it is, U's tower is now spoken for.

![Fifth Checkpoint](SS5.png)

### Checkpoint 6

We must reach `d3 #18 = 88` in 3 jumps, with simple addition each step.  There are a number of candidate paths, but importantly the first move must be `d7` or `g6`, since the other two paths starting with `h7` or `e6` quickly die.  This will be important later, as we will rule out `g6` elegantly.  At this point, I also started greying out cells that could not contain towers.

> `f8 #15 = 37` → `+16` → `(d7 or g6) #16 = 53` → `+17` → `#17 = 70` → `+18` → `d3 #18 = 88`

![Sixth Checkpoint](SS6.png)

### Checkpoint 7

Here is where the math gets a bit more sticky and harder to just try things out.  We no longer have the 3 move rule, but rather the `K` move rule, with `K` being something we need to find.  I played around for a while until finding a valid solution of `K = 7`, using `88 +19 +20 ×21 +22 +23 ÷24 +25 = 138`, which is at `f6`.  This took a while and while I was confident it was probably unique and correct, I wanted to make sure, so I had AI make a tool to do this search.  The condition that the path had to be between 4 and 9 steps in length and could only multiply or divide once in a row was restrictive, and the tool quickly found that this was the unique solution, so `K = 7`, and the future checkpoints fell out easily.

> `d3 #18 = 88` → `+19` → `#19 = 107` → `+20` → `#20 = 127` → `×21` → `▲ #21 = 2667`
> → `+22` → `▲ #22 = 2689` → `+23` → `▲ #23 = 2712` → `÷24` → `#24 = 113` → `+25` → `f6 #25 = 138`

Note the shape this forces: `×21` climbs onto a tower, `+22` and `+23` are level moves *at tower
altitude*, and `÷24` drops back down. **Moves 21, 22 and 23 are three towers in a row.**

![Seventh Checkpoint](SS7.png)

### The remaining checkpoints

With `K = 7` the late checkpoints are fixed at moves 25, 32, 39, 46, 53.  Using our tool, the solutions are unique (though the pathing is still very open).

| leg | arithmetic | towers used |
|---|---|---|
| #19–#25 → `f6` = 138 | `88 +19 +20 = 127`, `×21 = 2667`, `+22 = 2689`, `+23 = 2712`, `÷24 = 113`, `+25 = 138` | **3** |
| #26–#32 → `f3` = 272 | `138 +26 +27 +28 +29 = 248`, `×30 = 7440`, `÷31 = 240`, `+32 = 272` | 1 |
| #33–#39 → `b4` = 449 | `272 ×33 = 8976`, `÷34 = 264`, `+35 … +38 = 410`, `+39 = 449` | 1 |
| #40–#46 → `b3` = 750 | pure addition: `449 + (40+…+46) = 449 + 301 = 750` | 0 |
| #47–#53 → `h8` = 1100 | pure addition: `750 + (47+…+53) = 750 + 350 = 1100` | 0 |

Note that 5 additional towers are used, leaving only one tower unused after arriving at `1100`. We must hit that tower at some point.  We've now hit all checkpoints and know the exact move order we hit them in.  All that's left (and it's the crux of the puzzle) is to determine a legal path order.

![All Checkpoints](SS8.png)

### Crux 1: f7 and g6 are heavily restricted

Two strong restrictions allow us to make progress from this point.  First is the pathing from `d3` to `f6`.  We need to jump into `f6` from level ground after jumping off a tower that is the third tower in a row.  This pathing that requires 3 towers ends up heavily restrictive.  The towers can either occur in the bottom two regions and the Z pentomino in the top right.  Or, they can occur in the center region, the N pentomino, and the Z pentomino in the top right.  In the Z pentomino, the towers can be on `f7`, `g6`, or `h5` due to the requirement of landing a knight's move from `f6` after the jump.  However, `h5` can be ruled out because there is no valid tower square it can be jumped to from.  So there is a tower on either `f7` or `g6` during the `d3` to `f6` pathing!

#### Caveat: I missed a third family here

Ok, so I totally missed a third family here. Being honest about how this actually went — when
I solved it I only found the two families above. There is a third, which is wrong, but very hard to prove so naturally (I tried very hard). Enumerating every legal `d3 #18` → `f6 #25` chain gives **16 survivors**, in three region-triples rather than two:

| towers at `#21 #22 #23` | regions | ends on |
|---|---|---|
| `c4▲ → e5▲ → f7▲/g6▲` | N, O, Z | `f7` or `g6` |
| `g2▲ → h4▲ → f7▲/g6▲` | W, L, Z | `f7` or `g6` |
| **`h7▲ → g5▲ → h3▲`** (and its mirror) | **V, Z, L** | `h3` or `h7` |

I tried to prove it dead, and the closest I got is genuinely nice: in that family `h3` is L's tower, so the `×33` climb out of `f3` has to land on `f1`, which leaves only N and O for the `#30` tower — and of the three ways a tower in N or O
can drop into a knight's move of `f3`, the best one is blocked because `g5` is that family's own
Z tower and can't be stood on at ground level. That kills some of it, but not all: `d4▲ → d2`
survives, and so does `c5▲ → e5` in one of the two chains.  I couldn't figure out a way to dispel either without bifurcating.

#### Ok back to crux 1

Assuming that `f7` and `g6` must contain a tower... Those two cells are very interesting — they are the only cells reachable from `h8`!  Since `h8` is reached via a normal ground level knight's jump, one of `f7` and `g6` must be used to reach `h8`.  Thus, both `g6` and `f7` are accounted for!!
Concretely:

- `h8 #53 = 1100` is arrived at via `+53`, a **level** move into a ground square. Its predecessor
  `#52` must be a ground knight-neighbour of `h8`, and there are exactly two: **`f7` and `g6`**.
- Moves 21 through 23 jump from tower to tower to tower.  There are only a few valid ways in the grid to do this: `c4▲ #21` → `e5▲ #22` → `f7▲/g6▲ #23`, or `g2▲ #21` → `h4▲ #22` → `f7▲/g6▲ #23`.  Note that both end on either `f7` or `g6`.
- So `#23` ∈ {`f7▲`, `g6▲`} — region **Z**'s tower — and `#52` ∈ {`f7`, `g6`} on the ground.
  They are different squares, so **the path consumes both**.

**Key takeaway: no other part of the path can contain `f7` or `g6`.**

This has two major consequences.

First, the pathing from `f8 #15 = 37` to `d3 #18 = 88` on moves 16, 17 and 18 can no longer go through `g6`. It must therefore be:

> `f8 #15 = 37` → `+16` → `d7 #16 = 53` → `+17` → `c5 #17 = 70` → `+18` → `d3 #18 = 88`

Second, there is no longer a valid level knight's move *out* of `h8`.  Since we need to hit one more tower, we must jump to it immediately.  `f8` is already used at `#15`, so the only valid move is therefore to place a tower on `h6` and jump to it from `h8`, completing our route:

> `h8 #53 = 1100` → `×54` → `h6▲ #54 = 59400`

![After crux 1](SS9.png)

### Crux 2: which towers does the `d3` → `f6` path take?

To prove that `f7`/`g6` had a tower, we demonstrated that the `d3` → `f6` path must have towers at either `c4` and `e5` or at `g2` and `h4`.  We can determine that the `c4`/`e5` solution is correct by examining the `f3` → `b4` path.  Since that path immediately steps up onto a tower (`×33`), we must place a tower 2 cells away in a straight orthogonal direction from `f3`.  The candidates are `f5`, `f1` and `h3` — and `f5` is in region **F**, whose tower is already `e3`.  So the `#33` tower is `f1[W]` or `h3[L]`.  That removes `g2[W]` and `h4[L]` as valid pathing solutions for `d3` → `f6`, since they would consume both W and L and leave nothing for `#33`.  So the towers are at `c4` and `e5`.

Moreover, the only valid location to jump up to `c4` from is `a4`, which determines the pathing from `d3` to `e5` completely:

> `d3 #18 = 88` → `+19` → `b2 #19 = 107` → `+20` → `a4 #20 = 127` → `×21` → `c4▲ #21 = 2667` → `+22` → `e5▲ #22 = 2689`

![After crux 2](SS10.png)

### Endgame: I gave it to AI

The natural next place to look was which tower did the `f3 #32 = 272` jump up to.  The only two
candidates are `f1` and `h3`.

I worked on the rest for a long time — the stretch from `b3 #46` backwards to `b4 #39`, and the
seven level moves running out to `h8 #53` — and kept getting to the point where I had two
plausible branches and no clean way to choose between them. Eventually I just handed it to AI
to solve.

I don't feel too bad about that, because by then the problem space was *deeply* restricted, and
restricting it was the actual work:

- all twelve clue cells placed, at known move numbers, with `K = 7`
- every score from `#0` to `#53` pinned — each gap between checkpoints has exactly **one**
  arithmetic route, so the numbers were completely determined
- therefore every altitude known too, so the towers sat at `#0 #1 #2 #7 #12 #13 #14 #21 #22
  #23 #30 #33` and one more after `#53`
- nine of the thirteen towers already located, and most of the board greyed out as no-tower

All that was left was choosing 31 squares whose move types were already fixed. The search ran
in about **20 milliseconds** and came back with exactly one answer — which is a decent sign the
deductions above had done their job.

### The solution

![Solution](SS11.png)

#### Towers

| region | square | visited at |
|---|---|---|
| T | `a1` | #0 |
| X | `c2` | #1 |
| F | `e3` | #2 |
| P | `e6` | #7 |
| Y | `a5` | #12 |
| U | `b7` | #13 |
| I | `d8` | #14 |
| N | `c4` | #21 |
| O | `e5` | #22 |
| Z | `f7` | #23 |
| W | `e1` | #30 |
| L | `h3` | #33 |
| V | `h6` | #54 |

#### The path

| # | sq | op | score | | # | sq | op | score | | # | sq | op | score |
|--:|:--|:--|--:|-|--:|:--|:--|--:|-|--:|:--|:--|--:|
| 0 | `a1`▲ | — | **0** | | 19 | `b2` | `+19` | 107 | | 38 | `c6` | `+38` | 410 |
| 1 | `c2`▲ | `+1` | 1 | | 20 | `a4` | `+20` | 127 | | 39 | `b4` | `+39` | **449** |
| 2 | `e3`▲ | `+2` | 3 | | 21 | `c4`▲ | `×21` | 2667 | | 40 | `a6` | `+40` | 489 |
| 3 | `g3` | `÷3` | **1** | | 22 | `e5`▲ | `+22` | 2689 | | 41 | `c7` | `+41` | 530 |
| 4 | `h1` | `+4` | 5 | | 23 | `f7`▲ | `+23` | 2712 | | 42 | `b5` | `+42` | 572 |
| 5 | `f2` | `+5` | 10 | | 24 | `h7` | `÷24` | 113 | | 43 | `a3` | `+43` | 615 |
| 6 | `e4` | `+6` | **16** | | 25 | `f6` | `+25` | **138** | | 44 | `b1` | `+44` | 659 |
| 7 | `e6`▲ | `×7` | 112 | | 26 | `d5` | `+26` | 164 | | 45 | `d2` | `+45` | 704 |
| 8 | `e8` | `÷8` | 14 | | 27 | `c3` | `+27` | 191 | | 46 | `b3` | `+46` | **750** |
| 9 | `d6` | `+9` | **23** | | 28 | `a2` | `+28` | 219 | | 47 | `d4` | `+47` | 797 |
| 10 | `c8` | `+10` | 33 | | 29 | `c1` | `+29` | 248 | | 48 | `e2` | `+48` | 845 |
| 11 | `a7` | `+11` | 44 | | 30 | `e1`▲ | `×30` | 7440 | | 49 | `f4` | `+49` | 894 |
| 12 | `a5`▲ | `×12` | **528** | | 31 | `g1` | `÷31` | 240 | | 50 | `g2` | `+50` | 944 |
| 13 | `b7`▲ | `+13` | 541 | | 32 | `f3` | `+32` | **272** | | 51 | `h4` | `+51` | 995 |
| 14 | `d8`▲ | `+14` | 555 | | 33 | `h3`▲ | `×33` | 8976 | | 52 | `g6` | `+52` | 1047 |
| 15 | `f8` | `÷15` | **37** | | 34 | `h5` | `÷34` | 264 | | 53 | `h8` | `+53` | **1100** |
| 16 | `d7` | `+16` | 53 | | 35 | `g7` | `+35` | 299 | | 54 | `h6`▲ | `×54` | 59400 |
| 17 | `c5` | `+17` | 70 | | 36 | `f5` | `+36` | 335 | |  |  |  |  |
| 18 | `d3` | `+18` | **88** | | 37 | `e7` | `+37` | 372 | |  |  |  |  |

#### Unvisited squares

| square | on-path neighbours | sum |
|---|---|--:|
| `a8` | `a7`=44 | 44 |
| `b8` | `b7`=541 + `c8`=33 | 574 |
| `g8` | `g7`=299 + `f8`=37 + `h8`=1100 | 1436 |
| `b6` | `b7`=541 + `b5`=572 + `a6`=489 + `c6`=410 | 2012 |
| `g5` | `g6`=1047 + `f5`=335 + `h5`=264 | 1646 |
| `g4` | `g3`=1 + `f4`=894 + `h4`=995 | 1890 |
| `h2` | `h3`=8976 + `h1`=5 + `g2`=944 | 9925 |
| `d1` | `d2`=704 + `c1`=248 + `e1`=7440 | 8392 |
| `f1` | `f2`=10 + `e1`=7440 + `g1`=240 | 7690 |
| | **total** | **33609** |

`▲` marks a tower. Bold scores are the twelve printed clues, at moves 0, 3, 6, 9, 12, 15, 18,
25, 32, 39, 46, 53.

### Answer extraction

For each unvisited square, sum the scores of its orthogonally adjacent squares that lie on the
path; add those nine sums.

### **Answer: 33609**

### Tools

Built along the way, all self-contained HTML in this directory:

- **[`knight-sandbox.html`](knight-sandbox.html)** — the board. Place and remove towers, mark
  squares `?` / `✗ no tower`, walk the knight with legal moves highlighted and the resulting
  score previewed on hover. Runs can start anywhere, float without a move number until one is
  known, be built forwards *or backwards* (inverting each operation), and join up when they
  meet — sharing out their numbering.
- **[`score-paths.html`](score-paths.html)** — pure score arithmetic, no board. Given a score,
  a move number and an altitude, enumerate what is reachable, connect two scores in a fixed
  number of moves, or chain clue values checkpoint to checkpoint. This is what found `K = 7`.
- **[`print-grids.html`](print-grids.html)** — blank grids, four to a page.

Neither of those tools solves anything: they answer *"is this legal, and what does it score"*,
never *"what should I play next"*. The deductions above — the forced opening, `K = 7`, the
`f7`/`g6` squeeze, most of the tower placements — were made by hand, with the sandbox doing the
bookkeeping. The two places I handed over to a genuine search are both flagged where they
happen: killing off the last routes of the third tower family, and the endgame fill.

## The solving kit

| File | What it is |
|---|---|
| `knight-sandbox.html` | the interactive sandbox — double-click to open |
| `score-paths.html`    | score-arithmetic search: what can this score reach, and how |
| `print-grids.html`    | printable blank grids, 4 to a page |

No install, no build. If a yellow "autosave unavailable" strip appears, your browser blocks
storage on `file://` — either use **Export** to save by hand, or serve the folder:

    python3 -m http.server 8731
    # then open http://127.0.0.1:8731/knight-sandbox.html

### Modes

| Mode   | Key | What it does |
|--------|-----|--------------|
| Towers | `t` | Left-click toggles a tower. |
| Path   | `p` | Click a coloured dot to move the knight there. |
| Probe  | `b` | Inspect any square's reach without touching your work. |
| Erase  | `e` | Click a placed square to remove it. |

**Right-click any square** to cycle it `? maybe` → `✗ no tower` (drawn with a grey hatch) →
clear. The tower tracker then shows how many cells each region has left open.
**Shift-click** fills the pin form with that square.

### Keys

    ⌘Z / ⌘⇧Z        undo / redo (everything, not just moves)
    Backspace       drop the last move off the active segment
    r               flip between Forward and Backward
    t / p / b / e   switch mode

### Runs — start anywhere, join them up later

The trunk always starts at a1, move 0, score 0. Everything else is a **run** you can start,
extend and merge freely.

**Start one anywhere.** In Path mode, click any empty square. The run begins there with **no
move number** — it floats. Its shape and altitude are still checked, and it still refuses
revisits, but scores stay `?` until it gets a number. Squares in a floating run are labelled
`+0`, `+1`, `+2` rather than `#N`.

**Give it a number when you know one**, via *starts at move* in the Segments card, or unset it
to float again. Numbering a run whose first square is a clue cell picks up that clue as its
entry score automatically.

**Join two runs by pathing into each other.** When the run you're building can reach another
run's endpoint, that target is drawn with a **ring**. Click it and the two merge into one.
Whichever side already has move numbers hands them to the other — so a floating run walked out
from f6 gets real move numbers the moment the trunk reaches it, and every score in it
resolves. Going **Forward** you join onto another run's first square; going **Backward**, onto
its last. If both runs already have numbers they have to line up, otherwise the join isn't
offered.

You can still anchor a run at a known move directly: **add a segment** with a cell and a move
number. The move number is optional now — leave it blank and the run floats.

- **Forward** extends the head. **Backward** works back toward earlier moves, inverting each
  operation — a level arrival on move 18 into 138 means move 17 held 120; a ×18 arrival means
  138 must divide by 18. Illegal predecessors are refused with the reason.
- Every run renders on the board with its own colour stripe.
- Overlaps are flagged: the same square in two runs, or two runs claiming one move. Floating
  runs claim no move slots, so they can't collide on numbering.

### Removing things

- **Erase mode** (`e`) — click any placed square to remove just that one. At either end of a
  run it simply shortens; erase the anchor and the run re-anchors forward, picking up the new
  first square's clue value as its entry score if it has one. Erase from the **middle** and the
  run **splits in two**, with the tail becoming its own segment that keeps its real move
  numbers. To drop everything *after* a move instead, click that row in the move log.
- Squares that sit past a break in their run are drawn with a **red dashed outline and `#N?`**,
  so an orphaned tail is visible and can be erased rather than being invisible.
- **clear** on a segment row resets it to its anchor square, keeping the segment. **✕** deletes
  a segment outright. **Clear all path work** returns to just a1 while keeping towers and marks.
- **clear all** on the pins card drops every pin.
- a1 at move 0 is the fixed start and can't be erased; locked moves are protected until you
  unlock. Both say so rather than failing silently, and every removal is undoable with ⌘Z.

### The rest

- **Legal dots** are coloured by type: grey level (+N), blue up (×N), orange down (÷N). Hover
  one to preview the resulting score. Illegal targets say *why* — "already used",
  "78 is not divisible by 9", "a straight 2-step must change altitude by 1".
- **Lock** freezes a trunk prefix you trust; drop/rewind refuse to cross it until you unlock.
- **Pins** record "cell X is move N" as a pure annotation, and go red when a segment disagrees.
  The **seed** button turns a pin into a segment.
- **Notes** is a free-text box saved with your state and included in Export — handy if you're
  narrating as you go and want the commentary attached to the position it belongs to.
- **Checkpoint checks** (toolbar, off by default) verifies the recording schedule and narrows
  K. It does more of the deduction for you, hence off.
- **Answer** computes the neighbour-sum total from your squares, marked *provisional* until
  you have one unbroken path, 13 towers one per region, all visited.

There is no solver. The tool answers "is this legal, and what does it score" — never "what
should I play next".

### Printing

`print-grids.html` renders blank boards with the pentomino walls and clue numbers. Controls
(hidden when printing): grids per page (4 / 2 / 1), number of sheets, and toggles for clue
numbers, coordinates, a label line, and region shading. 4-up gives ~0.4" cells; drop to 2-up
if you want room to write four-digit scores.

### Score paths

Pure arithmetic, no board — it answers "from this score, what can I land on, and by what
sequence of operations". The altitude rule makes the operation order exact rather than a
guess: on **ground** you may only `+N` (stay) or `×N` (rise); on a **tower** only `+N` (stay)
or `÷N` (fall, when it divides evenly). So `×` and `÷` must alternate with any run of `+`
between them — never two multiplies or two divides in a row.

Four modes:

- **Explore** — from a score, the next move number, and an altitude, list everything reachable
  within *n* moves. Runs **forward or backward**: backward inverts each operation, so from 138
  arriving on move 18 it reports that a level arrival came from 120, and a `×18` arrival is
  impossible because 18 doesn't divide 138. Filter to clue values, to checkpoint moves, or to
  paths ending on the ground.
- **Connect** — "can score A at move i become score B at move j?" over a range of move counts.
  A negative answer is a real deduction: it rules the pairing out whatever the board looks
  like. Uses meet-in-the-middle, so wide gaps stay fast.
- **Ladder** — chain clue values checkpoint to checkpoint, each leg a fixed gap, no value
  reused. This is the "moves 1–18 at gap 3" shape of the puzzle.
- **Matrix** — for every ordered pair of clue values, how many `gap`-move sequences join them.

Two constraints beyond the arithmetic are baked in, and both prune hard:

- **Tower budget.** An altitude-1 state means standing on a tower, and there are only 13
  towers, so any path using more than 13 is impossible.
- **K is 4–9.** 12 clue cells = 7 early records (moves 0, 3, …, 18) plus 5 late ones, so the
  path reaches move 18+5K, and M ≤ 63 forces K ≤ 9. That makes "K unknown" a real filter
  rather than a free pass — move 19 can never be a checkpoint, because no K in 4–9 divides 1.
  Results show which K values each checkpoint is consistent with.

Results are candidates, not answers — a route that works arithmetically still has to be
realisable as knight moves. Check it in the sandbox.

### Self-tests

In the sandbox, run `kmTests(true)` in the console — 64 checks covering the board extraction,
move generation, altitude rules, the divisibility gate, forward and inverse scoring, run
anchoring, unknown entry scores, conflict detection, erasing (end, anchor, middle-split, lone
anchor, and the a1 and lock guards), floating runs, and joining in both directions including
the numbering handover and the mismatch refusal.

In score paths, press **run self-tests** (or call `spTests(true)`) — 38 checks. They include
exhaustive sweeps proving backward is the exact inverse of forward and that no backward step
invents an illegal forward move, a replay of every explored path, `connect` cross-checked
against brute-force enumeration, the tower cap, the move-63 ceiling, and the checkpoint
schedule including the unknown-K case.